# M5 Dataset Exploration and Analysis

This notebook provides comprehensive exploration of the M5 Walmart dataset.

## Dataset Overview
- **Source**: Walmart (M5 Forecasting Competition)
- **Products**: 3,049 items across 10 stores
- **Time Period**: 1,969 days (2011-01-29 to 2016-06-19)
- **Structure**: Hierarchical (State → Store → Category → Department → Item)
- **Features**: Sales, prices, calendar events, SNAP benefits

## Objectives
1. Understand data structure and quality
2. Identify patterns, seasonality, and trends
3. Detect potential causal relationships
4. Assess suitability for supply chain modeling
5. Define preprocessing requirements

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 6)

# Project imports
import sys
sys.path.append('../..')
from src.utils.config import get_config

config = get_config()
print("✓ Libraries loaded successfully")

## 1. Load Data

In [ ]:
# Data paths
data_path = config.get('data.raw_path')

# Load datasets
print("Loading datasets...")
calendar = pd.read_csv(f"{data_path}/calendar.csv")
sales = pd.read_csv(f"{data_path}/sales_train_validation.csv")
prices = pd.read_csv(f"{data_path}/sell_prices.csv")

print(f"\nCalendar shape: {calendar.shape}")
print(f"Sales shape: {sales.shape}")
print(f"Prices shape: {prices.shape}")

## 2. Data Structure Analysis

In [ ]:
# Calendar data
print("CALENDAR DATA")
print("="*50)
print(calendar.head())
print("\nColumns:", calendar.columns.tolist())
print("\nData types:")
print(calendar.dtypes)

In [ ]:
# Sales data structure
print("\nSALES DATA")
print("="*50)
print(sales.head())
print("\nHierarchical structure:")
print(f"States: {sales['state_id'].nunique()}")
print(f"Stores: {sales['store_id'].nunique()}")
print(f"Categories: {sales['cat_id'].nunique()}")
print(f"Departments: {sales['dept_id'].nunique()}")
print(f"Items: {sales['item_id'].nunique()}")
print(f"Total SKUs: {len(sales)}")

In [ ]:
# Price data
print("\nPRICE DATA")
print("="*50)
print(prices.head())
print("\nPrice statistics:")
print(prices['sell_price'].describe())

## 3. Time Series Transformation

In [ ]:
# Transform wide format to long format
print("Transforming sales data to long format...")

# Get day columns
day_cols = [col for col in sales.columns if col.startswith('d_')]
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

# Melt to long format
sales_long = sales.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='sales'
)

# Merge with calendar
sales_long = sales_long.merge(calendar, on='d', how='left')

# Convert date
sales_long['date'] = pd.to_datetime(sales_long['date'])

print(f"\nLong format shape: {sales_long.shape}")
print(sales_long.head())

## 4. Exploratory Data Analysis

In [ ]:
# Overall sales trends
daily_sales = sales_long.groupby('date')['sales'].sum().reset_index()

fig = px.line(
    daily_sales,
    x='date',
    y='sales',
    title='Total Daily Sales Across All Stores'
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Sales by state
state_sales = sales_long.groupby(['date', 'state_id'])['sales'].sum().reset_index()

fig = px.line(
    state_sales,
    x='date',
    y='sales',
    color='state_id',
    title='Daily Sales by State'
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Sales by category
cat_sales = sales_long.groupby(['date', 'cat_id'])['sales'].sum().reset_index()

fig = px.line(
    cat_sales,
    x='date',
    y='sales',
    color='cat_id',
    title='Daily Sales by Category'
)
fig.update_layout(height=400)
fig.show()

## 5. Seasonality Analysis

In [ ]:
# Add time features
daily_sales['year'] = daily_sales['date'].dt.year
daily_sales['month'] = daily_sales['date'].dt.month
daily_sales['day_of_week'] = daily_sales['date'].dt.dayofweek
daily_sales['week'] = daily_sales['date'].dt.isocalendar().week

# Weekly seasonality
dow_sales = daily_sales.groupby('day_of_week')['sales'].mean().reset_index()
dow_sales['day_name'] = dow_sales['day_of_week'].map({
    0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'
})

fig = px.bar(
    dow_sales,
    x='day_name',
    y='sales',
    title='Average Sales by Day of Week'
)
fig.show()

In [ ]:
# Monthly seasonality
monthly_sales = daily_sales.groupby('month')['sales'].mean().reset_index()
monthly_sales['month_name'] = monthly_sales['month'].map({
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
})

fig = px.bar(
    monthly_sales,
    x='month_name',
    y='sales',
    title='Average Sales by Month'
)
fig.show()

## 6. Event Analysis

In [ ]:
# Event impact
event_sales = sales_long.groupby(['date', 'event_name_1'])['sales'].sum().reset_index()
event_sales = event_sales[event_sales['event_name_1'].notna()]

print("Sales on event days:")
print(event_sales.groupby('event_name_1')['sales'].mean().sort_values(ascending=False))

## 7. Zero Sales and Intermittency Analysis

In [ ]:
# Calculate zero sales percentage by product
zero_sales = sales[day_cols].apply(lambda x: (x == 0).sum() / len(x) * 100, axis=1)

print("Zero sales statistics:")
print(zero_sales.describe())

# Plot distribution
fig = px.histogram(
    zero_sales,
    nbins=50,
    title='Distribution of Zero Sales Percentage by Product'
)
fig.update_xaxes(title='Percentage of Days with Zero Sales')
fig.update_yaxes(title='Number of Products')
fig.show()

## 8. Sample Product Deep Dive

In [ ]:
# Select a random product for detailed analysis
sample_id = sales['id'].iloc[100]
sample_data = sales_long[sales_long['id'] == sample_id].copy()

print(f"Analyzing product: {sample_id}")
print(f"Item: {sample_data['item_id'].iloc[0]}")
print(f"Store: {sample_data['store_id'].iloc[0]}")
print(f"Category: {sample_data['cat_id'].iloc[0]}")

# Plot sales
fig = px.line(
    sample_data,
    x='date',
    y='sales',
    title=f'Sales Time Series: {sample_id}'
)
fig.show()

## 9. Price Analysis

In [ ]:
# Price dynamics
item_sample = sample_data['item_id'].iloc[0]
store_sample = sample_data['store_id'].iloc[0]

# Get prices for sample item
item_prices = prices[
    (prices['item_id'] == item_sample) & 
    (prices['store_id'] == store_sample)
].copy()

# Merge with calendar
item_prices = item_prices.merge(
    calendar[['wm_yr_wk', 'date']].drop_duplicates(),
    on='wm_yr_wk'
)
item_prices['date'] = pd.to_datetime(item_prices['date'])

# Plot price changes
fig = px.line(
    item_prices,
    x='date',
    y='sell_price',
    title=f'Price Changes: {item_sample} at {store_sample}'
)
fig.show()

## 10. Key Insights and Next Steps

### Insights from EDA:
1. **Seasonality**: Strong weekly and monthly patterns
2. **Events**: Significant impact from holidays and special events
3. **Intermittency**: Many products have sparse sales
4. **Hierarchy**: Rich hierarchical structure (state/store/category/dept/item)
5. **Exogenous factors**: Price, SNAP, events affect demand

### Next Steps:
1. Build causal graph of demand drivers
2. Develop probabilistic forecasting models
3. Design simulation environment
4. Create multi-agent RL framework
5. Define evaluation metrics

In [ ]:
# Save processed data sample for quick testing
sample_size = 100
sampled_ids = sales['id'].sample(n=sample_size, random_state=42).tolist()

# Filter data
sales_sample = sales[sales['id'].isin(sampled_ids)]
sales_long_sample = sales_long[sales_long['id'].isin(sampled_ids)]

# Save
output_path = "../../data/processed/sample"
import os
os.makedirs(output_path, exist_ok=True)

sales_sample.to_csv(f"{output_path}/sales_sample.csv", index=False)
sales_long_sample.to_csv(f"{output_path}/sales_long_sample.csv", index=False)

print(f"\n✓ Sample data saved to {output_path}")
print(f"Sample contains {sample_size} products")